<a href="https://colab.research.google.com/github/miperezf/ChakraForm/blob/main/Reporte_Manzanas_ASOEX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Reporte Ejecutivo de Exportaciones — Manzanas Chile
**Fuente:** Frutas de Chile / ASOEX &nbsp;|&nbsp; **Base:** `BaseAsoex.xlsx`

> **Instrucciones:**
> 1. Sube `BaseAsoex.xlsx` al entorno de Colab (panel izquierdo → 📁 → subir)
> 2. Ajusta los parámetros en la **Celda 2** si es necesario
> 3. `Runtime → Run all`
> 4. El reporte HTML se descarga automáticamente al terminar


In [84]:
# ─────────────────────────────────────────────────────────────
# CELDA 1 · INSTALACIÓN (sin kaleido)
# ─────────────────────────────────────────────────────────────
!pip install -q openpyxl plotly


In [85]:
# ─────────────────────────────────────────────────────────────
# CELDA 2 · CONFIGURACIÓN GENERAL  ← ajustar aquí para otras especies
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import os
import datetime
from google.colab import files   # comentar si corres fuera de Colab

# ── SEMANA ACTUAL (automática) ────────────────────────────────
WK_ACTUAL = datetime.date.today().isocalendar()[1]
print(f"📅 Semana actual del año: {WK_ACTUAL}")

# ── PARÁMETROS AJUSTABLES ─────────────────────────────────────
ARCHIVO_XLSX     = "BaseAsoex.xlsx"
ESPECIE_OBJETIVO = "MANZANAS"
NOMBRE_ESPECIE   = "Manzanas"
AÑO_ACTUAL       = datetime.date.today().year        # ← también automático
AÑO_PREVIO       = AÑO_ACTUAL - 1                    # ← siempre el año anterior
TEMPORADA_ACTUAL = str(AÑO_ACTUAL)
TEMPORADA_PREVIA = str(AÑO_PREVIO)
ESTIMACION_TON   = 580_867     # ← toneladas totales estimadas para la temporada
COLOR_PRIMARIO   = "#1B4F72"
COLOR_SECUNDARIO = "#2E86C1"
COLOR_ACENTO     = "#28B463"
COLOR_NEGATIVO   = "#E74C3C"
COLOR_FONDO      = "#F0F4F8"
LOGO_NOMBRE      = "Frutas de Chile"

print(f"📊 Comparando {AÑO_PREVIO} vs {AÑO_ACTUAL} hasta semana {WK_ACTUAL}")
print("✅ Configuración cargada")

📅 Semana actual del año: 23
📊 Comparando 2025 vs 2026 hasta semana 23
✅ Configuración cargada


In [86]:
# ─────────────────────────────────────────────────────────────
# CELDA 3 · CARGA Y FILTRADO
# ─────────────────────────────────────────────────────────────
print(f"⏳ Cargando {ARCHIVO_XLSX}...")
df_raw = pd.read_excel(ARCHIVO_XLSX)
print(f"✅ Base completa: {len(df_raw):,} registros | {df_raw['Especie'].nunique()} especies")

df = df_raw[df_raw['Especie'] == ESPECIE_OBJETIVO].copy()
print(f"✅ {NOMBRE_ESPECIE}: {len(df):,} registros | Temporadas: {sorted(df['Temporada'].unique())}")


⏳ Cargando BaseAsoex.xlsx...
✅ Base completa: 373,902 registros | 64 especies
✅ Manzanas: 51,108 registros | Temporadas: ['2023-2024', '2024-2025', '2025-2026']


In [87]:
# ─────────────────────────────────────────────────────────────
# CELDA 4 · FUNCIONES AUXILIARES
# ─────────────────────────────────────────────────────────────
def fmt_miles(n):
    if pd.isna(n): return "-"
    return f"{int(n):,}".replace(",", ".")

def variacion_pct(nuevo, anterior):
    if anterior == 0: return None
    return round((nuevo - anterior) / anterior * 100, 1)

def color_var(v):
    if v is None: return "#888"
    return COLOR_ACENTO if v >= 0 else COLOR_NEGATIVO

def semanas_comunes(df_a, df_p):
    wk_a = df_a['Semana_'].str.extract(r'-(\d+)$')[0].astype(int).max()
    wk_p = df_p['Semana_'].str.extract(r'-(\d+)$')[0].astype(int).max()
    return min(wk_a, wk_p)

def fig_to_html_div(fig, div_id, height=400):
    """Convierte una figura Plotly a un <div> HTML interactivo.
    No requiere Kaleido ni ningún binario externo."""
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        div_id=div_id,
        config={"responsive": True, "displayModeBar": False},
        default_height=f"{height}px",
    )

MESES_ORDEN = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

print("✅ Funciones auxiliares definidas")


✅ Funciones auxiliares definidas


In [88]:
# ─────────────────────────────────────────────────────────────
# CELDA 5 · PREPARACIÓN DE TABLAS
# ─────────────────────────────────────────────────────────────
t_act  = df[df['Año'] == AÑO_ACTUAL]
t_prev = df[df['Año'] == AÑO_PREVIO]

# Semana máxima común (respeta la semana actual del calendario)
def wk_num(t):
    return t['Semana_'].str.extract(r'-(\d+)$')[0].astype(int).max()

wk_max = min(WK_ACTUAL, wk_num(t_act), wk_num(t_prev))
print(f"📅 Semana máxima común: {wk_max} (semana actual: {WK_ACTUAL})")

# Filtrar ambos años hasta wk_max
def filtrar_hasta_semana(t, wk):
    t2 = t.copy()
    t2['n_semana'] = t2['Semana_'].str.extract(r'-(\d+)$')[0].astype(int)
    return t2[t2['n_semana'] <= wk]

t_act  = filtrar_hasta_semana(t_act,  wk_max)
t_prev = filtrar_hasta_semana(t_prev, wk_max)
print(f"✅ Comparando {AÑO_PREVIO} vs {AÑO_ACTUAL} hasta semana {wk_max}")

# ── CAJAS EQUIVALENTES (normalizado a 19.5 kg) ────────────────
FACTOR_EQUIV = 19.5
t_act  = t_act.copy()
t_prev = t_prev.copy()
t_act['CajasEq']  = t_act['Kilos']  / FACTOR_EQUIV
t_prev['CajasEq'] = t_prev['Kilos'] / FACTOR_EQUIV
print(f"✅ Cajas equivalentes calculadas (factor: {FACTOR_EQUIV} kg)")

# 5.1 Totales
total_act  = int(t_act['CajasEq'].sum())
total_prev = int(t_prev['CajasEq'].sum())
var_total  = variacion_pct(total_act, total_prev)
kgs_act    = int(t_act['Kilos'].sum())
kgs_prev   = int(t_prev['Kilos'].sum())
var_kg     = variacion_pct(kgs_act, kgs_prev)
# ── ESTIMACIÓN TEMPORADA COMPLETA ─────────────────────────────
ESTIMACION_CAJAS = int(ESTIMACION_TON * 1_000 / FACTOR_EQUIV)   # ton → kg → cajas eq.
cajas_pendientes = max(0, ESTIMACION_CAJAS - total_act)
pct_pendiente    = round(cajas_pendientes / ESTIMACION_CAJAS * 100, 1)
pct_cargado      = round(100 - pct_pendiente, 1)
print(f"📊 Est. temporada : {fmt_miles(ESTIMACION_CAJAS)} cajas eq. ({ESTIMACION_TON:,} ton)".replace(",","."))
print(f"   Cargado        : {pct_cargado}%  ({fmt_miles(total_act)} cajas)")
print(f"   Por cargar     : {pct_pendiente}%  ({fmt_miles(cajas_pendientes)} cajas)")
print(f"📦 Total cajas eq. {AÑO_ACTUAL}: {fmt_miles(total_act)}  ({var_total:+.1f}% vs {AÑO_PREVIO})")

# 5.2 Por línea varietal
def tabla_linea(t):
    return t.groupby('Línea')['CajasEq'].sum().rename(str(t['Año'].iloc[0]))

lineas = pd.concat([tabla_linea(t_prev), tabla_linea(t_act)], axis=1).fillna(0)
lineas.columns = [str(AÑO_PREVIO), str(AÑO_ACTUAL)]
lineas['Var %'] = lineas.apply(
    lambda r: variacion_pct(r[str(AÑO_ACTUAL)], r[str(AÑO_PREVIO)]), axis=1)
lineas = lineas.sort_values(str(AÑO_ACTUAL), ascending=False)

# 5.3 Por región
def tabla_region(t):
    return t.groupby('Región')['CajasEq'].sum().rename(str(t['Año'].iloc[0]))

regiones = pd.concat([tabla_region(t_prev), tabla_region(t_act)], axis=1).fillna(0)
regiones.columns = [str(AÑO_PREVIO), str(AÑO_ACTUAL)]
regiones['Var %'] = regiones.apply(
    lambda r: variacion_pct(r[str(AÑO_ACTUAL)], r[str(AÑO_PREVIO)]), axis=1)
regiones = regiones.sort_values(str(AÑO_ACTUAL), ascending=False)

# 5.4 Por país (top 15)
paises = pd.DataFrame({
    str(AÑO_PREVIO): t_prev.groupby('País Destino')['CajasEq'].sum(),
    str(AÑO_ACTUAL): t_act.groupby('País Destino')['CajasEq'].sum(),
}).fillna(0).sort_values(str(AÑO_ACTUAL), ascending=False).head(15)
paises['Var %'] = paises.apply(
    lambda r: variacion_pct(r[str(AÑO_ACTUAL)], r[str(AÑO_PREVIO)]), axis=1)

# 5.5 Acumulado semanal
def semanas_temp(t_df):
    t2 = t_df.copy()
    t2['n_semana'] = t2['Semana_'].str.extract(r'-(\d+)$')[0].astype(int)
    return (t2.groupby('n_semana')['CajasEq']
              .sum().sort_index().cumsum().reset_index()
              .rename(columns={'n_semana': 'Semana', 'CajasEq': 'Acum'}))

sem_act_f  = semanas_temp(t_act)
sem_prev_f = semanas_temp(t_prev)

# 5.6 Por mes
mes_act  = t_act.groupby('Mes')['CajasEq'].sum().reindex(MESES_ORDEN, fill_value=0)
mes_prev = t_prev.groupby('Mes')['CajasEq'].sum().reindex(MESES_ORDEN, fill_value=0)

print("✅ Tablas preparadas")

📅 Semana máxima común: 22 (semana actual: 23)
✅ Comparando 2025 vs 2026 hasta semana 22
✅ Cajas equivalentes calculadas (factor: 19.5 kg)
📊 Est. temporada : 29.788.051 cajas eq. (580.867 ton)
   Cargado        : 48.0%  (14.310.758 cajas)
   Por cargar     : 52.0%  (15.477.293 cajas)
📦 Total cajas eq. 2026: 14.310.758  (+0.9% vs 2025)
✅ Tablas preparadas


In [89]:
# ─────────────────────────────────────────────────────────────
# CELDA 5b · VERIFICACIÓN DE CORTE SEMANAL
# ─────────────────────────────────────────────────────────────
print(f"📅 Semana máxima común usada: {wk_max}")
print(f"\n--- Semanas en {TEMPORADA_ACTUAL} ---")
print(sorted(t_act['n_semana'].unique()))
print(f"Max semana {TEMPORADA_ACTUAL}: {t_act['n_semana'].max()}")

print(f"\n--- Semanas en {TEMPORADA_PREVIA} ---")
print(sorted(t_prev['n_semana'].unique()))
print(f"Max semana {TEMPORADA_PREVIA}: {t_prev['n_semana'].max()}")

print(f"\n--- Total cajas por temporada ---")
print(f"{TEMPORADA_ACTUAL}:  {int(t_act['Cajas'].sum()):,}")
print(f"{TEMPORADA_PREVIA}: {int(t_prev['Cajas'].sum()):,}")

📅 Semana máxima común usada: 22

--- Semanas en 2026 ---
[np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]
Max semana 2026: 22

--- Semanas en 2025 ---
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]
Max semana 2025: 22

--- Total cajas por temporada ---
2026:  15,323,951
2025: 15,501,509


In [90]:
# ─────────────────────────────────────────────────────────────
# CELDA 6 · GENERACIÓN DE GRÁFICOS (Plotly → HTML div, sin Kaleido)
# ─────────────────────────────────────────────────────────────
LAYOUT = dict(
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#2C3E50", size=12),
    title_font=dict(size=14, color=COLOR_PRIMARIO),
    legend=dict(bgcolor="rgba(255,255,255,0.85)", bordercolor="#ddd", borderwidth=1),
    margin=dict(l=55, r=20, t=45, b=45),
    separators=",.",   # decimal="," miles="."  (formato chileno)
)

# G1: Acumulado semanal
fig_acum = go.Figure()
fig_acum.add_trace(go.Scatter(
    x=sem_prev_f['Semana'], y=sem_prev_f['Acum'],
    name=TEMPORADA_PREVIA, line=dict(color="#95A5A6", width=2, dash="dash"),
    fill="tozeroy", fillcolor="rgba(149,165,166,0.08)",
    hovertemplate="Semana %{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>",  # ← NUEVO
))
fig_acum.add_trace(go.Scatter(
    x=sem_act_f['Semana'], y=sem_act_f['Acum'],
    name=TEMPORADA_ACTUAL, line=dict(color=COLOR_SECUNDARIO, width=3),
    fill="tozeroy", fillcolor="rgba(46,134,193,0.10)",
    hovertemplate="Semana %{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>",  # ← NUEVO
))
fig_acum.update_layout(
    title=f"Cajas acumuladas por semana · {NOMBRE_ESPECIE}",
    xaxis_title="Semana de temporada", yaxis_title="Cajas acumuladas",
    yaxis_tickformat=",d", **LAYOUT
)
div_acum = fig_to_html_div(fig_acum, "g_acum", height=380)
print("✅ Gráfico 1: Acumulado semanal")

# G2: Barras mensuales
meses_con_datos = [m for m in MESES_ORDEN if mes_act[m] > 0 or mes_prev[m] > 0]
fig_mes = go.Figure([
    go.Bar(x=meses_con_datos, y=[mes_prev[m] for m in meses_con_datos],
           name=TEMPORADA_PREVIA, marker_color="#BDC3C7",
           hovertemplate="%{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>"),  # ← NUEVO
    go.Bar(x=meses_con_datos, y=[mes_act[m] for m in meses_con_datos],
           name=TEMPORADA_ACTUAL, marker_color=COLOR_SECUNDARIO,
           hovertemplate="%{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>"),  # ← NUEVO
])
fig_mes.update_layout(
    title=f"Despachos mensuales · {NOMBRE_ESPECIE}",
    barmode="group", yaxis_tickformat=",d", **LAYOUT
)
div_mes = fig_to_html_div(fig_mes, "g_mes", height=360)
print("✅ Gráfico 2: Mensual")

# G3: Dona por región
fig_reg = go.Figure(go.Pie(
    labels=regiones.index.tolist(),
    values=regiones[TEMPORADA_ACTUAL].tolist(),
    hole=0.44, textinfo="percent+label",
    marker=dict(colors=px.colors.qualitative.Bold),
    textfont=dict(size=11),
    hovertemplate="<b>%{label}</b><br>%{value:,.0f} cajas eq.<br>%{percent}<extra></extra>",
))
fig_reg.update_layout(
    title=f"Regiones · {TEMPORADA_ACTUAL}", showlegend=False,
    margin=dict(l=10, r=10, t=45, b=10), paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#2C3E50", size=11),
    title_font=dict(size=14, color=COLOR_PRIMARIO),
    separators=",.",
)
div_reg = fig_to_html_div(fig_reg, "g_reg", height=360)
print("✅ Gráfico 3: Regiones")

# G4: Barras horizontales por línea
lineas_plot = lineas[lineas.index != 'Sin Información'].sort_values(TEMPORADA_ACTUAL)
fig_lin = go.Figure([
    go.Bar(y=lineas_plot.index.tolist(), x=lineas_plot[TEMPORADA_PREVIA].tolist(),
           name=TEMPORADA_PREVIA, orientation='h', marker_color="#BDC3C7",
           hovertemplate="%{y}<br>%{fullData.name}: %{x:,.0f} cajas<extra></extra>"),  # ← NUEVO
    go.Bar(y=lineas_plot.index.tolist(), x=lineas_plot[TEMPORADA_ACTUAL].tolist(),
           name=TEMPORADA_ACTUAL, orientation='h', marker_color=COLOR_SECUNDARIO,
           hovertemplate="%{y}<br>%{fullData.name}: %{x:,.0f} cajas<extra></extra>"),  # ← NUEVO
])
fig_lin.update_layout(
    title="Líneas varietales", barmode="group", xaxis_tickformat=",d",
    margin=dict(l=160, r=20, t=45, b=45), paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#2C3E50", size=11),
    title_font=dict(size=14, color=COLOR_PRIMARIO),
    legend=dict(bgcolor="rgba(255,255,255,0.85)", bordercolor="#ddd", borderwidth=1),
    separators=",.",
)
div_lin = fig_to_html_div(fig_lin, "g_lin", height=380)
print("✅ Gráfico 4: Líneas varietales")

# G5: Top 10 países
top10 = paises.head(10)
fig_paises = go.Figure([
    go.Bar(x=top10.index.tolist(), y=top10[TEMPORADA_PREVIA].tolist(),
           name=TEMPORADA_PREVIA, marker_color="#BDC3C7",
           hovertemplate="%{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>"),  # ← NUEVO
    go.Bar(x=top10.index.tolist(), y=top10[TEMPORADA_ACTUAL].tolist(),
           name=TEMPORADA_ACTUAL, marker_color=COLOR_SECUNDARIO,
           hovertemplate="%{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>"),  # ← NUEVO
])
fig_paises.update_layout(
    title=f"Top 10 destinos · {NOMBRE_ESPECIE}",
    barmode="group", yaxis_tickformat=",d", **LAYOUT
)
div_paises = fig_to_html_div(fig_paises, "g_paises", height=380)
print("✅ Gráfico 5: Países")

✅ Gráfico 1: Acumulado semanal
✅ Gráfico 2: Mensual
✅ Gráfico 3: Regiones
✅ Gráfico 4: Líneas varietales
✅ Gráfico 5: Países


In [108]:
# ─────────────────────────────────────────────────────────────
# CELDA 7 · HELPERS HTML
# ─────────────────────────────────────────────────────────────
def fila_tabla(nombre, v_prev, v_act, var, destacar=False):
    v  = "" if var is None else f"{var:+.1f}%"
    c  = "inherit" if var is None else (COLOR_ACENTO if var >= 0 else COLOR_NEGATIVO)
    bg = f'style="background:{COLOR_FONDO};"' if destacar else ""
    return f"""<tr {bg}>
        <td>{nombre}</td>
        <td style="text-align:right">{fmt_miles(v_prev)}</td>
        <td style="text-align:right"><strong>{fmt_miles(v_act)}</strong></td>
        <td style="text-align:right;color:{c};font-weight:bold">{v}</td>
    </tr>"""

def build_tabla(df_t, excluir=None):
    rows = ""
    for i, (idx, row) in enumerate(df_t.iterrows()):
        if excluir and idx in excluir: continue
        rows += fila_tabla(idx, row[TEMPORADA_PREVIA], row[TEMPORADA_ACTUAL], row['Var %'], i%2==0)
    return rows

def kpi_card(titulo, v_act, v_prev, var, unidad="cajas", sufijo=""):
    vc = color_var(var)
    vs = "" if var is None else f"{'▲' if var >= 0 else '▼'} {abs(var):.1f}%"
    return f"""<div class="kpi-card">
        <div class="kpi-title">{titulo}</div>
        <div class="kpi-value">{fmt_miles(v_act)}{sufijo}</div>
        <div class="kpi-sub">{unidad}</div>
        <div class="kpi-var" style="color:{vc}">{vs}</div>
        <div class="kpi-ref">{TEMPORADA_PREVIA}: {fmt_miles(v_prev)}</div>
    </div>"""

thead_4 = lambda c1,c2,c3,c4: f"<thead><tr><th>{c1}</th><th style='text-align:right'>{c2}</th><th style='text-align:right'>{c3}</th><th style='text-align:right'>{c4}</th></tr></thead>"

print("✅ Helpers HTML definidos")


✅ Helpers HTML definidos


In [109]:
# ─────────────────────────────────────────────────────────────
# CELDA 8 · CONSTRUCCIÓN DEL HTML FINAL
# ─────────────────────────────────────────────────────────────

PLOTLYJS_CDN = "https://cdn.plot.ly/plotly-2.26.0.min.js"

html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Reporte · {NOMBRE_ESPECIE} · {AÑO_ACTUAL} · Wk{wk_max}</title>
<script src="{PLOTLYJS_CDN}"></script>
<style>
  *, *::before, *::after {{ box-sizing: border-box; }}
  body {{
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #E8EDF2;
    color: #2C3E50;
    margin: 0;
    padding: 24px 0;
    font-size: 14px;
  }}
  .page {{
    max-width: 1100px;
    margin: 0 auto;
    background: white;
    border-radius: 10px;
    overflow: hidden;
    box-shadow: 0 4px 24px rgba(0,0,0,.14);
  }}
  .header {{
    background: {COLOR_PRIMARIO};
    color: white;
    padding: 28px 40px 22px;
  }}
  .header-top {{ display:flex; justify-content:space-between; align-items:flex-start; }}
  .header h1  {{ margin:0 0 4px; font-size:26px; }}
  .header .sub {{ font-size:12px; opacity:.78; margin:3px 0 0; }}
  .header-badges {{ display:flex; gap:12px; }}
  .header-badge {{
    background: rgba(255,255,255,.18);
    border: 1px solid rgba(255,255,255,.3);
    border-radius: 6px;
    padding: 8px 18px;
    text-align: center;
    font-size: 12px;
    white-space: nowrap;
  }}
  .header-badge strong {{ display:block; font-size:20px; margin-top:2px; }}
  .divider-line {{
    height: 4px;
    background: linear-gradient(90deg, {COLOR_SECUNDARIO}, {COLOR_ACENTO}, #F39C12);
  }}
  .section {{ padding:26px 40px; border-bottom:1px solid #ECF0F1; }}
  .section:last-child {{ border-bottom:none; }}
  .section-title {{
    font-size:15px; font-weight:700; color:{COLOR_PRIMARIO};
    margin:0 0 16px; padding-bottom:8px;
    border-bottom:2px solid {COLOR_SECUNDARIO};
  }}
  .kpi-row {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(190px,1fr)); gap:16px; }}
  .kpi-card {{
    background:{COLOR_FONDO}; border-radius:10px;
    padding:18px 16px 14px; border-top:4px solid {COLOR_SECUNDARIO};
    text-align:center;
  }}
  .kpi-title {{ font-size:11px; text-transform:uppercase; letter-spacing:.8px; color:#7F8C8D; font-weight:600; }}
  .kpi-value {{ font-size:26px; font-weight:800; color:{COLOR_PRIMARIO}; margin:6px 0 2px; }}
  .kpi-sub   {{ font-size:10px; color:#95A5A6; margin-bottom:6px; }}
  .kpi-var   {{ font-size:17px; font-weight:700; }}
  .kpi-ref   {{ font-size:10px; color:#AAA; margin-top:4px; }}
  .chart-row {{
    display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-top:12px;
  }}
  .chart-box {{
    background:{COLOR_FONDO}; border-radius:10px; padding:12px; min-height:380px;
  }}
  table {{ width:100%; border-collapse:collapse; font-size:13px; }}
  thead {{ background:{COLOR_PRIMARIO}; color:white; }}
  thead th {{ padding:10px 12px; text-align:left; font-size:12px; letter-spacing:.4px; }}
  tbody tr td {{ padding:8px 12px; }}
  tbody tr:hover {{ background:rgba(46,134,193,.07) !important; }}
  .nota {{
    background:#FEF9E7; border-left:4px solid #F39C12;
    border-radius:0 8px 8px 0; padding:12px 16px;
    font-size:12px; color:#7D6608; margin-top:14px;
  }}
  .footer {{
    background:{COLOR_PRIMARIO}; color:rgba(255,255,255,.55);
    text-align:center; padding:16px; font-size:11px;
  }}
  @media print {{
    body {{ background:white; padding:0; }}
    .page {{ box-shadow:none; }}
  }}
</style>
</head>
<body>
<div class="page">

  <div class="header">
    <div class="header-top">
      <div>
        <h1>📊 Reporte Ejecutivo · {NOMBRE_ESPECIE}</h1>
        <p class="sub">Estadística de Exportaciones · Chile · {AÑO_ACTUAL}</p>
        <p class="sub">Fuente: {LOGO_NOMBRE} / ASOEX &nbsp;·&nbsp; Generado: {datetime.datetime.now().strftime('%d/%m/%Y %H:%M')}</p>
      </div>
      <div class="header-badges">
        <div class="header-badge">
          Año en curso<br>
          <strong>{AÑO_ACTUAL}</strong>
          vs {AÑO_PREVIO}
        </div>
        <div class="header-badge">
          Semana en curso<br>
          <strong>Wk {wk_max}</strong>
          acumulado a wk {wk_max}
        </div>
      </div>
    </div>
  </div>
  <div class="divider-line"></div>

  <div class="section">
    <div class="section-title">🏆 Indicadores Clave de Desempeño</div>
    <div class="kpi-row">
      {kpi_card("Total Cajas Eq. 19.5 kg", total_act, total_prev, var_total, "cajas eq. 19.5 kg")}
      {kpi_card("Total Kilos Exportados", kgs_act, kgs_prev, var_kg, "kg")}
      {kpi_card("Nº de Destinos", t_act['País Destino'].nunique(), t_prev['País Destino'].nunique(),
                variacion_pct(t_act['País Destino'].nunique(), t_prev['País Destino'].nunique()), "países")}
      {kpi_card("Por cargar · est. temporada", pct_pendiente, None, None,
                f"{fmt_miles(cajas_pendientes)} cajas eq. restantes", sufijo="%")}
    </div>
  </div>

  <div class="section">
    <div class="section-title">📈 Evolución Acumulada por Semana</div>
    {div_acum}
    <div class="nota">
      ⚠️ Comparación hasta semana {wk_max} del año.
      Variación acumulada: <strong style="color:{color_var(var_total)}">{var_total:+.1f}%</strong>
      respecto al mismo período de {AÑO_PREVIO}. Cifras en cajas eq. 19.5 kg.
    </div>
  </div>

  <div class="section">
    <div class="section-title">📅 Despachos Mensuales · cajas eq. 19.5 kg</div>
    {div_mes}
  </div>

  <div class="section">
    <div class="section-title">🌍 Distribución Regional y Varietal · cajas eq. 19.5 kg</div>
    <div class="chart-row">
      <div class="chart-box">{div_reg}</div>
      <div class="chart-box">{div_lin}</div>
    </div>
  </div>

  <div class="section">
    <div class="section-title">🌍 Detalle por Región</div>
    <table>
      {thead_4("Región", f"{AÑO_PREVIO} (cajas eq. 19.5 kg)", f"{AÑO_ACTUAL} (cajas eq. 19.5 kg)", "Variación %")}
      <tbody>{build_tabla(regiones)}</tbody>
    </table>
  </div>

  <div class="section">
    <div class="section-title">🍎 Detalle por Línea Varietal</div>
    <table>
      {thead_4("Línea Varietal", f"{AÑO_PREVIO} (cajas eq. 19.5 kg)", f"{AÑO_ACTUAL} (cajas eq. 19.5 kg)", "Variación %")}
      <tbody>{build_tabla(lineas, excluir=['Sin Información'])}</tbody>
    </table>
  </div>

  <div class="section">
    <div class="section-title">🏳️ Top 10 Países de Destino · cajas eq. 19.5 kg</div>
    {div_paises}
  </div>

  <div class="section">
    <div class="section-title">📋 Ranking de Destinos (Top 15)</div>
    <table>
      {thead_4("País Destino", f"{AÑO_PREVIO} (cajas eq. 19.5 kg)", f"{AÑO_ACTUAL} (cajas eq. 19.5 kg)", "Variación %")}
      <tbody>{build_tabla(paises)}</tbody>
    </table>
  </div>

  <div class="footer">
    Reporte generado automáticamente · {LOGO_NOMBRE} / ASOEX ·
    Datos: Base oficial de exportaciones de Chile ·
    {datetime.datetime.now().strftime('%B %Y')} · <em>Para uso interno</em>
  </div>

</div>
</body>
</html>"""

print("✅ HTML construido correctamente")

✅ HTML construido correctamente


In [110]:
# ─────────────────────────────────────────────────────────────
# CELDA 8b · DATOS LATINOAMÉRICA
# ─────────────────────────────────────────────────────────────

REGION_FOCO    = "LATINOAMERICA"
VPC_EMPRESAS   = ["POLAR FRUIT INT", "FRUTAM LTDA."]
VPC_NOMBRE     = "VPC"
COLOR_VPC      = "#F39C12"   # naranja destacado para VPC
COLOR_COMP     = "#BDC3C7"   # gris para competencia

# ── Filtrar región ────────────────────────────────────────────
la_act  = t_act[t_act['Región'] == REGION_FOCO].copy()
la_prev = t_prev[t_prev['Región'] == REGION_FOCO].copy()

total_la_act  = la_act['CajasEq'].sum()
total_la_prev = la_prev['CajasEq'].sum()
var_la        = variacion_pct(total_la_act, total_la_prev)
print(f"📦 Latinoamérica {AÑO_ACTUAL}: {fmt_miles(total_la_act)} cajas eq. ({var_la:+.1f}%)")

# ── Ranking exportadores ──────────────────────────────────────
def ranking_exportadores(t_a, t_p):
    r_act  = t_a.groupby('Exportador')['CajasEq'].sum()
    r_prev = t_p.groupby('Exportador')['CajasEq'].sum()
    rank = pd.DataFrame({
        str(AÑO_PREVIO): r_prev,
        str(AÑO_ACTUAL):  r_act,
    }).fillna(0).sort_values(str(AÑO_ACTUAL), ascending=False)
    rank['Var %']      = rank.apply(lambda r: variacion_pct(r[str(AÑO_ACTUAL)], r[str(AÑO_PREVIO)]), axis=1)
    rank['Part % act'] = (rank[str(AÑO_ACTUAL)]  / total_la_act  * 100).round(1)
    rank['Part % prev']= (rank[str(AÑO_PREVIO)] / total_la_prev * 100).round(1)
    rank['Var part']   = (rank['Part % act'] - rank['Part % prev']).round(1)
    return rank

ranking = ranking_exportadores(la_act, la_prev)

# ── Agrupar VPC ───────────────────────────────────────────────
vpc_act   = la_act[la_act['Exportador'].isin(VPC_EMPRESAS)]['CajasEq'].sum()
vpc_prev  = la_prev[la_prev['Exportador'].isin(VPC_EMPRESAS)]['CajasEq'].sum()
vpc_part_act  = round(vpc_act  / total_la_act  * 100, 1)
vpc_part_prev = round(vpc_prev / total_la_prev * 100, 1)
vpc_var   = variacion_pct(vpc_act, vpc_prev)
vpc_var_part  = round(vpc_part_act - vpc_part_prev, 1)
print(f"🟠 VPC {AÑO_ACTUAL}: {fmt_miles(vpc_act)} cajas eq. | {vpc_part_act}% participación ({vpc_var:+.1f}%)")

# ── Acumulado semanal VPC vs Total LA ─────────────────────────
def sem_la(t, empresas=None):
    t2 = t.copy()
    if empresas:
        t2 = t2[t2['Exportador'].isin(empresas)]
    t2['n_semana'] = t2['Semana_'].str.extract(r'-(\d+)$')[0].astype(int)
    return (t2.groupby('n_semana')['CajasEq']
              .sum().sort_index().cumsum().reset_index()
              .rename(columns={'n_semana': 'Semana', 'CajasEq': 'Acum'}))

sem_la_act_total = sem_la(la_act)
sem_la_prev_total= sem_la(la_prev)
sem_la_act_vpc   = sem_la(la_act,  VPC_EMPRESAS)
sem_la_prev_vpc  = sem_la(la_prev, VPC_EMPRESAS)

# ── Top países dentro de Latinoamérica ───────────────────────
paises_la = pd.DataFrame({
    str(AÑO_PREVIO): la_prev.groupby('País Destino')['CajasEq'].sum(),
    str(AÑO_ACTUAL):  la_act.groupby('País Destino')['CajasEq'].sum(),
}).fillna(0).sort_values(str(AÑO_ACTUAL), ascending=False).head(10)
paises_la['Var %']     = paises_la.apply(lambda r: variacion_pct(r[str(AÑO_ACTUAL)], r[str(AÑO_PREVIO)]), axis=1)
paises_la['Part % act']= (paises_la[str(AÑO_ACTUAL)] / total_la_act * 100).round(1)

# ── Desglose varietal en Latinoamérica ────────────────────────
lineas_la = pd.DataFrame({
    str(AÑO_PREVIO): la_prev.groupby('Línea')['CajasEq'].sum(),
    str(AÑO_ACTUAL):  la_act.groupby('Línea')['CajasEq'].sum(),
}).fillna(0).sort_values(str(AÑO_ACTUAL), ascending=False)
lineas_la['Var %']     = lineas_la.apply(lambda r: variacion_pct(r[str(AÑO_ACTUAL)], r[str(AÑO_PREVIO)]), axis=1)
lineas_la['Part % act']= (lineas_la[str(AÑO_ACTUAL)] / total_la_act * 100).round(1)

print("✅ Datos Latinoamérica preparados")

📦 Latinoamérica 2026: 6.663.874 cajas eq. (+7.7%)
🟠 VPC 2026: 705.795 cajas eq. | 10.6% participación (+11.6%)
✅ Datos Latinoamérica preparados


In [111]:
# ─────────────────────────────────────────────────────────────
# CELDA 8c · GRÁFICOS LATINOAMÉRICA
# ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# G6: Acumulado semanal VPC vs Total Latinoamérica
fig_la_acum = go.Figure()
fig_la_acum.add_trace(go.Scatter(
    x=sem_la_prev_total['Semana'], y=sem_la_prev_total['Acum'],
    name=f"Total LA {AÑO_PREVIO}", line=dict(color="#BDC3C7", width=2, dash="dash"),
    fill="tozeroy", fillcolor="rgba(189,195,199,0.08)",
    hovertemplate="Semana %{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>",
))
fig_la_acum.add_trace(go.Scatter(
    x=sem_la_act_total['Semana'], y=sem_la_act_total['Acum'],
    name=f"Total LA {AÑO_ACTUAL}", line=dict(color=COLOR_SECUNDARIO, width=2),
    fill="tozeroy", fillcolor="rgba(46,134,193,0.08)",
    hovertemplate="Semana %{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>",
))
fig_la_acum.add_trace(go.Scatter(
    x=sem_la_prev_vpc['Semana'], y=sem_la_prev_vpc['Acum'],
    name=f"VPC {AÑO_PREVIO}", line=dict(color="#F0B27A", width=2, dash="dash"),
    hovertemplate="Semana %{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>",
))
fig_la_acum.add_trace(go.Scatter(
    x=sem_la_act_vpc['Semana'], y=sem_la_act_vpc['Acum'],
    name=f"VPC {AÑO_ACTUAL}", line=dict(color=COLOR_VPC, width=3),
    hovertemplate="Semana %{x}<br>%{fullData.name}: %{y:,.0f} cajas<extra></extra>",
))
fig_la_acum.update_layout(
    title=f"Acumulado semanal · Latinoamérica · VPC vs Total",
    xaxis_title="Semana", yaxis_title="Cajas eq. acumuladas",
    yaxis_tickformat=",d", **LAYOUT
)
div_la_acum = fig_to_html_div(fig_la_acum, "g_la_acum", height=400)
print("✅ Gráfico 6: Acumulado semanal LA")

# G7: Dona participación de mercado
# ── CORRECCIÓN: VPC siempre entra completo, top_n es solo para competidores ──
top_n = 8   # top N de competidores NO-VPC que se muestran por nombre

rank_vpc    = ranking[ranking.index.isin(VPC_EMPRESAS)]           # siempre incluidas
rank_no_vpc = ranking[~ranking.index.isin(VPC_EMPRESAS)]          # el resto

# Unir: empresas VPC (todas) + top N competidores
rank_plot = pd.concat([rank_vpc, rank_no_vpc.head(top_n)])

# Calcular Resto con lo que queda fuera
resto_act = total_la_act - rank_plot[str(AÑO_ACTUAL)].sum()
if resto_act > 0:
    rank_plot.loc['Resto'] = {
        str(AÑO_ACTUAL): resto_act, str(AÑO_PREVIO): 0,
        'Var %': None, 'Part % act': round(resto_act / total_la_act * 100, 1),
        'Part % prev': 0, 'Var part': 0
    }

# Construir listas para el donut agrupando VPC en una sola tajada
labels_dona   = []
values_dona   = []
colores_dona2 = []
vpc_sumado    = False

for exp in rank_plot.index:
    if exp in VPC_EMPRESAS:
        if not vpc_sumado:
            # Sumar TODAS las empresas VPC (incluso las que no estaban en top_n)
            total_vpc_donut = float(
                ranking.loc[ranking.index.isin(VPC_EMPRESAS), str(AÑO_ACTUAL)].sum()
            )
            labels_dona.append(VPC_NOMBRE)
            values_dona.append(total_vpc_donut)
            colores_dona2.append(COLOR_VPC)
            vpc_sumado = True
    else:
        labels_dona.append(exp)
        values_dona.append(float(rank_plot.loc[exp, str(AÑO_ACTUAL)]))
        colores_dona2.append("#ECF0F1" if exp == 'Resto' else COLOR_COMP)

fig_dona = go.Figure(go.Pie(
    labels=labels_dona, values=values_dona,
    hole=0.44, textinfo="percent+label",
    marker=dict(colors=colores_dona2),
    textfont=dict(size=11),
    pull=[0.08 if l == VPC_NOMBRE else 0 for l in labels_dona],
    hovertemplate="<b>%{label}</b><br>%{value:,.0f} cajas eq.<br>%{percent}<extra></extra>",
    customdata=[int(v) for v in values_dona],
))
fig_dona.update_layout(
    title=f"Participación de mercado · Latinoamérica · {AÑO_ACTUAL}",
    showlegend=False,
    margin=dict(l=10, r=10, t=45, b=10), paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#2C3E50", size=11),
    title_font=dict(size=14, color=COLOR_PRIMARIO),
    separators=",.",
)
div_dona = fig_to_html_div(fig_dona, "g_dona", height=400)
print("✅ Gráfico 7: Dona participación")

# Verificación rápida en consola
vpc_val_donut = values_dona[labels_dona.index(VPC_NOMBRE)]
print(f"\n🔎 Verificación donut vs banner:")
print(f"   VPC en donut : {vpc_val_donut:,.0f} cajas = {round(vpc_val_donut/total_la_act*100,1)}%".replace(",","."))
print(f"   VPC en banner: {vpc_act:,.0f} cajas = {vpc_part_act}%".replace(",","."))
print(f"   {'✅ Coinciden' if abs(vpc_val_donut - vpc_act) < 1 else '⚠️  Aún difieren'}")

✅ Gráfico 6: Acumulado semanal LA
✅ Gráfico 7: Dona participación

🔎 Verificación donut vs banner:
   VPC en donut : 705.795 cajas = 10.6%
   VPC en banner: 705.795 cajas = 10.6%
   ✅ Coinciden


In [112]:
# ─────────────────────────────────────────────────────────────
# CELDA 8d · HTML REPORTE LATINOAMÉRICA
# ─────────────────────────────────────────────────────────────

def fila_ranking(pos, nombre, v_prev, v_act, var, part_act, part_prev, var_part, es_vpc=False):
    v    = "" if var is None else f"{var:+.1f}%"
    cv   = "inherit" if var is None else (COLOR_ACENTO if var >= 0 else COLOR_NEGATIVO)
    vp   = "" if var_part is None else f"{var_part:+.1f}pp"
    cvp  = "inherit" if var_part is None else (COLOR_ACENTO if var_part >= 0 else COLOR_NEGATIVO)
    bold = 'style="background:#FEF9E7; font-weight:700;"' if es_vpc else ('style="background:#F0F4F8;"' if pos % 2 == 0 else "")
    badge= f'<span style="background:{COLOR_VPC};color:white;border-radius:4px;padding:1px 7px;font-size:10px;margin-left:6px;">VPC</span>' if es_vpc else ""
    return f"""<tr {bold}>
        <td style="text-align:center;color:#7F8C8D;font-size:12px">{pos}</td>
        <td>{nombre}{badge}</td>
        <td style="text-align:right">{fmt_miles(v_prev)}</td>
        <td style="text-align:right"><strong>{fmt_miles(v_act)}</strong></td>
        <td style="text-align:right;color:{cv};font-weight:bold">{v}</td>
        <td style="text-align:right"><strong>{part_act}%</strong></td>
        <td style="text-align:right;color:{cvp}">{vp}</td>
    </tr>"""

def fila_pais_la(pos, nombre, v_prev, v_act, var, part_act, destacar=False):
    v  = "" if var is None else f"{var:+.1f}%"
    cv = "inherit" if var is None else (COLOR_ACENTO if var >= 0 else COLOR_NEGATIVO)
    bg = 'style="background:#F0F4F8;"' if destacar else ""
    return f"""<tr {bg}>
        <td style="text-align:center;color:#7F8C8D;font-size:12px">{pos}</td>
        <td>{nombre}</td>
        <td style="text-align:right">{fmt_miles(v_prev)}</td>
        <td style="text-align:right"><strong>{fmt_miles(v_act)}</strong></td>
        <td style="text-align:right;color:{cv};font-weight:bold">{v}</td>
        <td style="text-align:right"><strong>{part_act}%</strong></td>
    </tr>"""

# ── Fila VPC consolidada (siempre primera) ────────────────────
filas_ranking = ""
filas_ranking += fila_ranking(
    "★", VPC_NOMBRE,
    vpc_prev, vpc_act, vpc_var,
    vpc_part_act, vpc_part_prev, vpc_var_part,
    es_vpc=True
)

# ── Top 20 competidores excluyendo empresas VPC ───────────────
pos = 1
for exp, row in ranking.iterrows():
    if exp in VPC_EMPRESAS:
        continue
    if pos > 20:
        break
    filas_ranking += fila_ranking(
        pos, exp,
        row[str(AÑO_PREVIO)], row[str(AÑO_ACTUAL)], row['Var %'],
        row['Part % act'], row['Part % prev'], row['Var part'],
        es_vpc=False
    )
    pos += 1

# ── Filas top países ──────────────────────────────────────────
filas_paises_la = ""
for i, (pais, row) in enumerate(paises_la.iterrows()):
    filas_paises_la += fila_pais_la(
        i+1, pais,
        row[str(AÑO_PREVIO)], row[str(AÑO_ACTUAL)], row['Var %'],
        row['Part % act'], i % 2 == 0
    )

# ── Filas líneas varietales ───────────────────────────────────
filas_lineas_la = ""
for i, (linea, row) in enumerate(lineas_la.iterrows()):
    if linea == 'Sin Información': continue
    v  = "" if row['Var %'] is None else f"{row['Var %']:+.1f}%"
    cv = "inherit" if row['Var %'] is None else (COLOR_ACENTO if row['Var %'] >= 0 else COLOR_NEGATIVO)
    bg = 'style="background:#F0F4F8;"' if i % 2 == 0 else ""
    filas_lineas_la += f"""<tr {bg}>
        <td>{linea}</td>
        <td style="text-align:right">{fmt_miles(row[str(AÑO_PREVIO)])}</td>
        <td style="text-align:right"><strong>{fmt_miles(row[str(AÑO_ACTUAL)])}</strong></td>
        <td style="text-align:right;color:{cv};font-weight:bold">{v}</td>
        <td style="text-align:right"><strong>{row['Part % act']}%</strong></td>
    </tr>"""

html_la = f"""
  <div class="header" style="background:{COLOR_PRIMARIO};">
    <div class="header-top">
      <div>
        <h1>🌎 Zoom Latinoamérica · {NOMBRE_ESPECIE}</h1>
        <p class="sub">Participación de Mercado y Competencia · {AÑO_ACTUAL}</p>
        <p class="sub">Fuente: {LOGO_NOMBRE} / ASOEX &nbsp;·&nbsp; Generado: {datetime.datetime.now().strftime('%d/%m/%Y %H:%M')}</p>
      </div>
      <div class="header-badges">
        <div class="header-badge">
          Año en curso<br>
          <strong>{AÑO_ACTUAL}</strong>
          vs {AÑO_PREVIO}
        </div>
        <div class="header-badge">
          Semana en curso<br>
          <strong>Wk {wk_max}</strong>
          acumulado a wk {wk_max}
        </div>
        <div class="header-badge" style="border-color:{COLOR_VPC};background:rgba(243,156,18,0.25);">
          Grupo VPC {AÑO_ACTUAL}<br>
          <strong>{vpc_part_act}%</strong>
          participación
        </div>
      </div>
    </div>
  </div>
  <div class="divider-line"></div>

  <div class="section">
    <div class="section-title">🏆 Indicadores · Latinoamérica</div>
    <div class="kpi-row">
      {kpi_card("Total Cajas Eq. 19.5 kg", int(total_la_act), int(total_la_prev), var_la, "cajas eq. 19.5 kg")}
      {kpi_card("VPC Cajas Eq. 19.5 kg", int(vpc_act), int(vpc_prev), vpc_var, "cajas eq. 19.5 kg")}
      {kpi_card("Part. VPC Latinoamérica", vpc_part_act, vpc_part_prev, vpc_var_part, "% del mercado")}
      {kpi_card("Países Destino", la_act['País Destino'].nunique(), la_prev['País Destino'].nunique(),
                variacion_pct(la_act['País Destino'].nunique(), la_prev['País Destino'].nunique()), "países Latinoamérica")}
    </div>
  </div>

  <div class="section">
    <div class="section-title">📈 Evolución Acumulada · VPC vs Total Latinoamérica</div>
    {div_la_acum}
    <div class="nota">
      ⚠️ Comparación hasta semana {wk_max} del año {AÑO_ACTUAL}.
      VPC acumula <strong>{fmt_miles(int(vpc_act))}</strong> cajas eq. 19.5 kg,
      representando el <strong style="color:{COLOR_VPC}">{vpc_part_act}%</strong>
      del mercado latinoamericano en {AÑO_ACTUAL}
      (vs {vpc_part_prev}% en {AÑO_PREVIO}).
    </div>
  </div>

  <div class="section">
    <div class="section-title">🥧 Participación de Mercado · Latinoamérica {AÑO_ACTUAL}</div>
    {div_dona}
  </div>

  <div class="section">
    <div class="section-title">🏆 Ranking Exportadores · Latinoamérica (Top 20)</div>
    <table>
      <thead><tr>
        <th style="text-align:center">#</th>
        <th>Exportador</th>
        <th style="text-align:right">{AÑO_PREVIO} (cajas eq. 19.5 kg)</th>
        <th style="text-align:right">{AÑO_ACTUAL} (cajas eq. 19.5 kg)</th>
        <th style="text-align:right">Var %</th>
        <th style="text-align:right">Part % {AÑO_ACTUAL}</th>
        <th style="text-align:right">Var part</th>
      </tr></thead>
      <tbody>{filas_ranking}</tbody>
    </table>
  </div>

  <div class="section">
    <div class="section-title">🌍 Top 10 Países · Latinoamérica</div>
    <table>
      <thead><tr>
        <th style="text-align:center">#</th>
        <th>País Destino</th>
        <th style="text-align:right">{AÑO_PREVIO} (cajas eq. 19.5 kg)</th>
        <th style="text-align:right">{AÑO_ACTUAL} (cajas eq. 19.5 kg)</th>
        <th style="text-align:right">Var %</th>
        <th style="text-align:right">Part % {AÑO_ACTUAL}</th>
      </tr></thead>
      <tbody>{filas_paises_la}</tbody>
    </table>
  </div>

  <div class="section">
    <div class="section-title">🍎 Líneas Varietales · Latinoamérica</div>
    <table>
      <thead><tr>
        <th>Línea Varietal</th>
        <th style="text-align:right">{AÑO_PREVIO} (cajas eq. 19.5 kg)</th>
        <th style="text-align:right">{AÑO_ACTUAL} (cajas eq. 19.5 kg)</th>
        <th style="text-align:right">Var %</th>
        <th style="text-align:right">Part % {AÑO_ACTUAL}</th>
      </tr></thead>
      <tbody>{filas_lineas_la}</tbody>
    </table>
  </div>

  <div class="footer">
    Reporte generado automáticamente · {LOGO_NOMBRE} / ASOEX ·
    Datos: Base oficial de exportaciones de Chile ·
    {datetime.datetime.now().strftime('%B %Y')} · <em>Para uso interno</em>
  </div>
"""

print("✅ HTML Latinoamérica construido")

✅ HTML Latinoamérica construido


In [113]:
# ─────────────────────────────────────────────────────────────
# CELDA 9 · CONSOLIDAR, GUARDAR Y DESCARGAR
# ─────────────────────────────────────────────────────────────

PLOTLYJS_CDN = "https://cdn.plot.ly/plotly-2.26.0.min.js"

html_final = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Reporte · {NOMBRE_ESPECIE} · {AÑO_ACTUAL} · Wk{wk_max}</title>
<script src="{PLOTLYJS_CDN}"></script>
<style>
  *, *::before, *::after {{ box-sizing: border-box; }}
  body {{
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #E8EDF2;
    color: #2C3E50;
    margin: 0;
    padding: 0;
    font-size: 14px;
  }}

  /* ── PESTAÑAS ── */
  .tabs-wrapper {{
    position: sticky;
    top: 0;
    z-index: 100;
    background: {COLOR_PRIMARIO};
    display: flex;
    gap: 4px;
    padding: 10px 40px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2);
  }}
  .tab-btn {{
    background: rgba(255,255,255,0.15);
    color: rgba(255,255,255,0.75);
    border: none;
    border-radius: 8px 8px 0 0;
    padding: 10px 28px;
    font-size: 13px;
    font-weight: 600;
    cursor: pointer;
    transition: all 0.2s;
    letter-spacing: 0.3px;
  }}
  .tab-btn:hover {{
    background: rgba(255,255,255,0.25);
    color: white;
  }}
  .tab-btn.active {{
    background: white;
    color: {COLOR_PRIMARIO};
  }}
  .tab-btn.active.vpc {{
    color: {COLOR_VPC};
  }}

  /* ── CONTENIDO ── */
  .tab-content {{ display: none; padding: 24px 0; }}
  .tab-content.active {{ display: block; }}
  .page {{
    max-width: 1100px;
    margin: 0 auto;
    background: white;
    border-radius: 10px;
    overflow: hidden;
    box-shadow: 0 4px 24px rgba(0,0,0,.14);
  }}

  /* ── HEADER ── */
  .header {{
    background: {COLOR_PRIMARIO};
    color: white;
    padding: 28px 40px 22px;
  }}
  .header-top {{ display:flex; justify-content:space-between; align-items:flex-start; }}
  .header h1  {{ margin:0 0 4px; font-size:26px; }}
  .header .sub {{ font-size:12px; opacity:.78; margin:3px 0 0; }}
  .header-badges {{ display:flex; gap:12px; }}
  .header-badge {{
    background: rgba(255,255,255,.18);
    border: 1px solid rgba(255,255,255,.3);
    border-radius: 6px;
    padding: 8px 18px;
    text-align: center;
    font-size: 12px;
    white-space: nowrap;
  }}
  .header-badge strong {{ display:block; font-size:20px; margin-top:2px; }}
  .divider-line {{
    height: 4px;
    background: linear-gradient(90deg, {COLOR_SECUNDARIO}, {COLOR_ACENTO}, #F39C12);
  }}

  /* ── SECCIONES ── */
  .section {{ padding:26px 40px; border-bottom:1px solid #ECF0F1; }}
  .section:last-child {{ border-bottom:none; }}
  .section-title {{
    font-size:15px; font-weight:700; color:{COLOR_PRIMARIO};
    margin:0 0 16px; padding-bottom:8px;
    border-bottom:2px solid {COLOR_SECUNDARIO};
  }}

  /* ── KPIs ── */
  .kpi-row {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(190px,1fr)); gap:16px; }}
  .kpi-card {{
    background:{COLOR_FONDO}; border-radius:10px;
    padding:18px 16px 14px; border-top:4px solid {COLOR_SECUNDARIO};
    text-align:center;
  }}
  .kpi-title {{ font-size:11px; text-transform:uppercase; letter-spacing:.8px; color:#7F8C8D; font-weight:600; }}
  .kpi-value {{ font-size:26px; font-weight:800; color:{COLOR_PRIMARIO}; margin:6px 0 2px; }}
  .kpi-sub   {{ font-size:10px; color:#95A5A6; margin-bottom:6px; }}
  .kpi-var   {{ font-size:17px; font-weight:700; }}
  .kpi-ref   {{ font-size:10px; color:#AAA; margin-top:4px; }}

  /* ── GRÁFICOS ── */
  .chart-row {{
    display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-top:12px;
  }}
  .chart-box {{
    background:{COLOR_FONDO}; border-radius:10px; padding:12px; min-height:380px;
  }}

  /* ── TABLAS ── */
  table {{ width:100%; border-collapse:collapse; font-size:13px; }}
  thead {{ background:{COLOR_PRIMARIO}; color:white; }}
  thead th {{ padding:10px 12px; text-align:left; font-size:12px; letter-spacing:.4px; }}
  tbody tr td {{ padding:8px 12px; }}
  tbody tr:hover {{ background:rgba(46,134,193,.07) !important; }}

  /* ── NOTA ── */
  .nota {{
    background:#FEF9E7; border-left:4px solid #F39C12;
    border-radius:0 8px 8px 0; padding:12px 16px;
    font-size:12px; color:#7D6608; margin-top:14px;
  }}

  /* ── FOOTER ── */
  .footer {{
    background:{COLOR_PRIMARIO}; color:rgba(255,255,255,.55);
    text-align:center; padding:16px; font-size:11px;
  }}

  @media print {{
    body {{ background:white; }}
    .tabs-wrapper {{ display:none; }}
    .tab-content {{ display:block !important; }}
    .page {{ box-shadow:none; margin-bottom:40px; }}
  }}
</style>
</head>
<body>

  <!-- PESTAÑAS -->
  <div class="tabs-wrapper">
    <button class="tab-btn active"     onclick="showTab('tab1', this)">📊 Resumen General</button>
    <button class="tab-btn vpc"        onclick="showTab('tab2', this)">🌎 Zoom Latinoamérica</button>
  </div>

  <!-- PESTAÑA 1: RESUMEN GENERAL -->
  <div id="tab1" class="tab-content active">
    <div class="page">
      {html}
    </div>
  </div>

  <!-- PESTAÑA 2: LATINOAMÉRICA -->
  <div id="tab2" class="tab-content">
    <div class="page">
      {html_la}
    </div>
  </div>

  <script>
    function showTab(tabId, btn) {{
      document.querySelectorAll('.tab-content').forEach(t => t.classList.remove('active'));
      document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
      document.getElementById(tabId).classList.add('active');
      btn.classList.add('active');
      window.scrollTo({{top: 0, behavior: 'smooth'}});
    }}
  </script>

</body>
</html>"""

# Guardar y descargar
nombre_archivo = f"Reporte_{NOMBRE_ESPECIE}_{AÑO_ACTUAL}_Wk{wk_max}.html"
with open(nombre_archivo, "w", encoding="utf-8") as f:
    f.write(html_final)

print(f"\n✅ Reporte consolidado: {nombre_archivo}")
print(f"   Tamaño: {os.path.getsize(nombre_archivo)/1024:.0f} KB")
print("⬇️  Iniciando descarga...")
files.download(nombre_archivo)
print("✅ ¡Listo! Revisa tu carpeta de descargas.")


✅ Reporte consolidado: Reporte_Manzanas_2026_Wk22.html
   Tamaño: 102 KB
⬇️  Iniciando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ ¡Listo! Revisa tu carpeta de descargas.
